# Rides API Request

**General Notebook Explanation**

- This notebook is used to retrieve initial public transportation trip data from the Open Bus Stride API (https://open-bus-stride-api.hasadna.org.il/docs#/
).

- The data extraction is limited to trips that either start or end within Tel Aviv.

- Users can define a custom start and end datetime range, allowing flexible querying of the dataset.

- The output of this notebook is a DataFrame containing the raw trip data, which serves as the foundation for subsequent processing and analysis steps in later stages of the workflow.

## Libraries 

In [1]:
import requests
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime, timezone, timedelta
from pathlib import Path
import time
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.time_utils import to_israel, fmt_time, fmt_date, day_he, round_hour, dur_min
from src.api_utils import fetch_with_retry,fetch_all_rides, fetch_stops, build_route_cache
from src.geo_utils import haversine
from src.config import BASE, TZ
from src.data_preparation import build_rows

!pip install requests pandas tqdm

## Config

In [12]:
FROM_DATE= "2024-04-06"
FROM_HOUR= "21"
TO_DATE= "2024-04-06"
TO_HOUR= "21"

OUT_FILE   = Path(f"../data/interim_data/telaviv_ride_from_{FROM_DATE}-{FROM_HOUR}00_to_{TO_DATE}-{str(int(TO_HOUR)+1)}00.csv")

## Process

In [4]:
### Fetach all rides
rides = fetch_all_rides(FROM_DATE,FROM_HOUR,TO_DATE,TO_HOUR)

Total 126 rides


100%|██████████| 126/126 [00:02<00:00, 44.56נסיעות/s]

✅ 126 Rides pulled


In [5]:
### Build the route cache
route_cache = build_route_cache(rides)

Pulling station to -99 routes unique...


100%|██████████| 99/99 [02:49<00:00,  1.72s/it]

✅ cache ready: 99 routes


In [10]:
### Create the rides Datafram
df_rides = build_rows(rides, route_cache)

## Export results

In [13]:
df_rides.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")
print(f"✅ Saved: {OUT_FILE.resolve()}")

✅ Saved: C:\Users\shaha\Documents\DS_course\public_transport_ML\data\interim_data\telaviv_ride_from_2024-04-06-2100_to_2024-04-06-2200.csv
